In [12]:
# Import necessary libraries
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
import transformers
import torch.nn as nn

# Load the dataset
dataset = load_dataset("CodeHima/TOS_DatasetV3")

dataset


DatasetDict({
    train: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 7945
    })
    validation: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 1050
    })
    test: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 1045
    })
})

In [13]:
# Convert to a pandas DataFrame for easier inspection
# We're assuming a 'train' split exists — we'll verify this below
df = pd.DataFrame(dataset['train'])

# Basic shape check — should show (rows, columns)
print(df.shape)

# Peek at the first few rows to see what we're working with
print(df.head())

# Check label distribution — important for knowing if we have a class imbalance problem
# e.g. if 90% of clauses are "clearly fair" the model will just learn to always predict that
print(df['sentence'].value_counts())

(7945, 2)
                                            sentence    unfairness_level
0  these terms and any rights and licenses grante...        clearly_fair
1  the user is responsible for all damages liabil...      clearly_unfair
2  no refunds for downtime  the company is not li...  potentially_unfair
3  ea recommends that parents and guardians famil...        clearly_fair
4  the company can limit or restrict your ability...  potentially_unfair
sentence
                                                                                                                                                                                                                    59
limitation of liability                                                                                                                                                                                              4
indemnity                                                                                                        

In [14]:
print(df['unfairness_level'].value_counts())

unfairness_level
clearly_fair          4044
potentially_unfair    1953
clearly_unfair        1948
Name: count, dtype: int64


In [15]:
# Check for nulls
print(df.isnull().sum())

# Check for rows where sentence is suspiciously short (like that "59" row)
print(df[df['sentence'].str.len() < 20])

sentence            0
unfairness_level    0
dtype: int64
              sentence unfairness_level
11            feedback     clearly_fair
13     user guidelines   clearly_unfair
45            services     clearly_fair
47                         clearly_fair
51    terms of service   clearly_unfair
...                ...              ...
7863          accurate     clearly_fair
7868                ix   clearly_unfair
7884           licence     clearly_fair
7920        assignment     clearly_fair
7929   billing support   clearly_unfair

[475 rows x 2 columns]


In [16]:
# Remove rows where sentence is less than 20 characters
# These are clearly fragments/garbage and will hurt training
df_clean = df[df['sentence'].str.len() >= 20].copy()

# Also strip any leading/trailing whitespace from sentences
df_clean['sentence'] = df_clean['sentence'].str.strip()

# Remove any rows that are empty after stripping
df_clean = df_clean[df_clean['sentence'].str.len() > 0]

# Sanity check — see how many rows survived
print(f"Original: {len(df)} rows")
print(f"Cleaned: {len(df_clean)} rows")
print(f"Dropped: {len(df) - len(df_clean)} rows")

Original: 7945 rows
Cleaned: 7470 rows
Dropped: 475 rows


In [17]:
from sklearn.model_selection import train_test_split

# Map text labels to integers so the model can work with them
# We'll keep this mapping saved so we can decode predictions later
label_map = {
    'clearly_fair': 0,
    'potentially_unfair': 1,
    'clearly_unfair': 2
}

# Apply the mapping to create a numeric label column
df_clean['label'] = df_clean['unfairness_level'].map(label_map)

# First split off 20% for test set
df_train_val, df_test = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=17,        # for reproducibility
    stratify=df_clean['label']  # maintains class distribution across splits
)

# Then split the remaining 80% into train (87.5%) and validation (12.5%)
# which gives us roughly 70/15/15 overall
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.15,
    random_state=17,
    stratify=df_train_val['label']
)

# Sanity check the sizes
print(f"Train: {len(df_train)} rows")
print(f"Validation: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

Train: 5079 rows
Validation: 897 rows
Test: 1494 rows


In [18]:
# Run this cell to save the pandas dfs to csv so they can be used in the other notebooks

df_train.to_csv('df_train.csv', index=False)
df_val.to_csv('df_val.csv', index=False)
df_test.to_csv('df_test.csv', index=False)